<a href="https://colab.research.google.com/github/Kishoby/Conceptual-Research_Hybrid-Approach/blob/Humidity/Humidity_3_Iterations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
import pandas as pd
import numpy as np

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("/content/drive/MyDrive/Research v2/Research 18.03.2026/cleaned_weather_data (1).csv")

print("Dataset loaded successfully!")
print(df.shape)

df.head()

Dataset loaded successfully!
(130003, 43)


,country,location_name,latitude,longitude,timezone,last_updated_epoch,last_updated,temperature_celsius,temperature_fahrenheit,condition_text,...,air_quality_us-epa-index,air_quality_gb-defra-index,sunrise,sunset,moonrise,moonset,moon_phase,moon_illumination,condition_text_before,wind_direction_before
0,United States of America,Washington Park,46.60,-120.49,America/Los_Angeles,1715849100,2024-05-16 01:45:00,16.1,61.0,2,...,1,1,05:26 AM,08:31 PM,01:36 PM,02:52 AM,Waxing Gibbous,55,Clear,SW
1,Honduras,Tegucigalpa,14.10,-87.22,America/Tegucigalpa,1715849100,2024-05-16 02:45:00,23.0,73.4,32,...,2,2,05:22 AM,06:09 PM,12:52 PM,12:55 AM,Waxing Gibbous,55,Partly cloudy,WSW
2,El Salvador,San Salvador,13.71,-89.20,America/El_Salvador,1715849100,2024-05-16 02:45:00,26.0,78.8,23,...,2,2,05:30 AM,06:16 PM,01:00 PM,01:02 AM,Waxing Gibbous,55,Moderate or heavy rain with thunder,S
3,Guatemala,Guatemala City,14.62,-90.53,America/Guatemala,1715849100,2024-05-16 02:45:00,20.0,68.0,19,...,4,10,05:34 AM,06:23 PM,01:05 PM,01:09 AM,Waxing Gibbous,55,Mist,S
4,Belize,Belmopan,17.25,-88.77,America/Belize,1715849100,2024-05-16 02:45:00,26.0,78.9,30,...,1,1,05:23 AM,06:20 PM,12:56 PM,01:04 AM,Waxing Gibbous,55,Overcast,E


In [15]:
import pandas as pd

# -----------------------------
# TARGET
# -----------------------------
target = "humidity"

# -----------------------------
# 1. REMOVE UNSUITABLE FEATURES
# -----------------------------
# Reasons:
# - duplicate units
# - weak usefulness / near-zero correlation
# - timestamp less interpretable unless converted to datetime features
# - remove duplicate-index style features if you want cleaner predictors

remove_features = [
    "temperature_fahrenheit",        # duplicate of temperature_celsius
    "wind_kph",                      # duplicate of wind_mph
    "pressure_in",                   # duplicate of pressure_mb
    "precip_in",                     # duplicate of precip_mm
    "feels_like_fahrenheit",         # duplicate of feels_like_celsius
    "visibility_miles",              # duplicate of visibility_km
    "gust_kph",                      # duplicate of gust_mph
    "moon_illumination",             # almost no correlation
    "wind_direction",                # very weak correlation
    "pressure_mb",                   # almost no correlation
    "last_updated_epoch"             # use only if converted into useful datetime features
]

# drop only if they exist
existing_remove_features = [col for col in remove_features if col in df.columns]

print("Features removed:")
print(existing_remove_features)

df_clean = df.drop(columns=existing_remove_features, errors="ignore")

# -----------------------------
# 2. SELECT MOST SUITABLE 15 FEATURES
# -----------------------------
# Chosen based on:
# - stronger correlation with humidity
# - no duplicate-unit columns
# - no target leakage
selected_features = [
    "uv_index",
    "cloud",
    "condition_text",
    "air_quality_Ozone",
    "temperature_celsius",
    "feels_like_celsius",
    "air_quality_us-epa-index",
    "air_quality_gb-defra-index",
    "longitude",
    "precip_mm",
    "air_quality_PM10",
    "air_quality_PM2.5",
    "visibility_km",
    "air_quality_Sulphur_dioxide",
    "latitude"
]

# keep only features that actually exist
selected_features = [col for col in selected_features if col in df_clean.columns]

print("\nSelected suitable features for humidity:")
print(selected_features)

# -----------------------------
# 3. FINAL DATASET FOR humidity
# -----------------------------
X_humidity = df_clean[selected_features]
y_humidity = df_clean[target]

final_humidity_df = pd.concat([X_humidity, y_humidity], axis=1)

print("\nFinal dataset shape:")
print(final_humidity_df.shape)

print("\nFinal dataset columns:")
print(final_humidity_df.columns.tolist())

# preview
final_humidity_df.head()

Features removed:
['temperature_fahrenheit', 'wind_kph', 'pressure_in', 'precip_in', 'feels_like_fahrenheit', 'visibility_miles', 'gust_kph', 'moon_illumination', 'wind_direction', 'pressure_mb', 'last_updated_epoch']

Selected suitable features for humidity:
['uv_index', 'cloud', 'condition_text', 'air_quality_Ozone', 'temperature_celsius', 'feels_like_celsius', 'air_quality_us-epa-index', 'air_quality_gb-defra-index', 'longitude', 'precip_mm', 'air_quality_PM10', 'air_quality_PM2.5', 'visibility_km', 'air_quality_Sulphur_dioxide', 'latitude']

Final dataset shape:
(130003, 16)

Final dataset columns:
['uv_index', 'cloud', 'condition_text', 'air_quality_Ozone', 'temperature_celsius', 'feels_like_celsius', 'air_quality_us-epa-index', 'air_quality_gb-defra-index', 'longitude', 'precip_mm', 'air_quality_PM10', 'air_quality_PM2.5', 'visibility_km', 'air_quality_Sulphur_dioxide', 'latitude', 'humidity']


,uv_index,cloud,condition_text,air_quality_Ozone,temperature_celsius,feels_like_celsius,air_quality_us-epa-index,air_quality_gb-defra-index,longitude,precip_mm,air_quality_PM10,air_quality_PM2.5,visibility_km,air_quality_Sulphur_dioxide,latitude,humidity
0,1.0,0,2,62.2,16.1,16.1,1,1,-120.49,0.00,7.1,6.3,16.0,0.2,46.60,58
1,1.0,37,32,23.3,23.0,25.3,2,2,-87.22,0.28,25.3,19.0,10.0,1.4,14.10,78
2,1.0,50,23,5.9,26.0,30.2,2,2,-89.20,0.30,28.1,20.4,10.0,7.5,13.71,94
3,1.0,100,19,0.4,20.0,20.0,4,10,-90.53,0.09,178.1,132.0,5.0,19.3,14.62,88
4,1.0,94,30,34.0,26.0,29.6,1,1,-88.77,0.00,32.1,7.7,10.0,0.2,17.25,89


In [16]:
removed_table = pd.DataFrame({
    "Removed Features": existing_remove_features
})

selected_table = pd.DataFrame({
    "Selected Features for PM2.5": selected_features
})

print("Removed Features Table")
display(removed_table)

print("Selected Features Table")
display(selected_table)

Removed Features Table


,Removed Features
0,temperature_fahrenheit
1,wind_kph
2,pressure_in
3,precip_in
4,feels_like_fahrenheit
5,visibility_miles
6,gust_kph
7,moon_illumination
8,wind_direction
9,pressure_mb


Selected Features Table


,Selected Features for PM2.5
0,uv_index
1,cloud
2,condition_text
3,air_quality_Ozone
4,temperature_celsius
5,feels_like_celsius
6,air_quality_us-epa-index
7,air_quality_gb-defra-index
8,longitude
9,precip_mm


In [17]:
from google.colab import files

# Save humidity dataset
final_humidity_df.to_csv("final_humidity_dataset.csv", index=False)

# Download to your computer
files.download("final_humidity_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [18]:
import pandas as pd
import numpy as np

# -----------------------------
# TARGET AND SELECTED FEATURES
# -----------------------------
target = "humidity"

selected_features = [
    "uv_index",
    "cloud",
    "condition_text",
    "air_quality_Ozone",
    "temperature_celsius",
    "feels_like_celsius",
    "air_quality_us-epa-index",
    "air_quality_gb-defra-index",
    "longitude",
    "precip_mm",
    "air_quality_PM10",
    "air_quality_PM2.5",
    "visibility_km",
    "air_quality_Sulphur_dioxide",
    "latitude"
]

# -----------------------------
# BUILD DATASET
# -----------------------------
humidity_df = df[["last_updated"] + selected_features + [target]].copy()

# convert datetime
humidity_df["last_updated"] = pd.to_datetime(humidity_df["last_updated"], errors="coerce")

# remove invalid dates
humidity_df = humidity_df.dropna(subset=["last_updated"])

# sort by time (VERY important for time-series / online learning)
humidity_df = humidity_df.sort_values("last_updated").reset_index(drop=True)

# drop remaining missing values
humidity_df = humidity_df.dropna().reset_index(drop=True)

print("Humidity dataset shape:", humidity_df.shape)
humidity_df.head()

Humidity dataset shape: (130003, 17)


,last_updated,uv_index,cloud,condition_text,air_quality_Ozone,temperature_celsius,feels_like_celsius,air_quality_us-epa-index,air_quality_gb-defra-index,longitude,precip_mm,air_quality_PM10,air_quality_PM2.5,visibility_km,air_quality_Sulphur_dioxide,latitude,humidity
0,2024-05-16 01:45:00,1.0,0,2,62.2,16.1,16.1,1,1,-120.49,0.00,7.1,6.3,16.0,0.2,46.60,58
1,2024-05-16 02:45:00,1.0,37,32,23.3,23.0,25.3,2,2,-87.22,0.28,25.3,19.0,10.0,1.4,14.10,78
2,2024-05-16 02:45:00,1.0,50,23,5.9,26.0,30.2,2,2,-89.20,0.30,28.1,20.4,10.0,7.5,13.71,94
3,2024-05-16 02:45:00,1.0,100,19,0.4,20.0,20.0,4,10,-90.53,0.09,178.1,132.0,5.0,19.3,14.62,88
4,2024-05-16 02:45:00,1.0,94,30,34.0,26.0,29.6,1,1,-88.77,0.00,32.1,7.7,10.0,0.2,17.25,89


In [19]:
# -----------------------------
# 80% TRAIN / 20% TEST
# -----------------------------
split_index = int(0.8 * len(humidity_df))

train_df = humidity_df.iloc[:split_index].copy()
test_df = humidity_df.iloc[split_index:].copy()

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)

Train shape: (104002, 17)
Test shape : (26001, 17)


In [20]:
from google.colab import files

# Save training dataset
train_df.to_csv("humidity_training_dataset.csv", index=False)

# Save testing dataset
test_df.to_csv("humidity_testing_dataset.csv", index=False)

# Download both files
files.download("humidity_training_dataset.csv")
files.download("humidity_testing_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [21]:
# -----------------------------
# ITERATION 1, 2, 3 FROM TRAINING DATA
# -----------------------------
n_train = len(train_df)

iter1 = train_df.iloc[:int(0.33 * n_train)].copy()
iter2 = train_df.iloc[int(0.33 * n_train):int(0.66 * n_train)].copy()
iter3 = train_df.iloc[int(0.66 * n_train):].copy()

print("Iteration 1 shape:", iter1.shape)
print("Iteration 2 shape:", iter2.shape)
print("Iteration 3 shape:", iter3.shape)

Iteration 1 shape: (34320, 17)
Iteration 2 shape: (34321, 17)
Iteration 3 shape: (35361, 17)


In [22]:
def get_xy(data):
    X = data[selected_features]
    y = data[target]
    return X, y

def calculate_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = mse ** 0.5
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    accuracy = r2 * 100
    return mse, rmse, mae, r2, accuracy

In [23]:
X1, y1 = get_xy(iter1)
X2, y2 = get_xy(iter2)
X3, y3 = get_xy(iter3)

X_test, y_test = get_xy(test_df)

print(X1.shape, X2.shape, X3.shape, X_test.shape)

(34320, 15) (34321, 15) (35361, 15) (26001, 15)


In [24]:
results = []            # for iteration-wise RMSE comparison table
all_iteration_metrics = []  # for full training iteration metrics of each model
final_test_results = [] # for final unseen test results

In [25]:
from sklearn.linear_model import SGDRegressor

scaler_sgd = StandardScaler()

X1_sgd = scaler_sgd.fit_transform(X1)
X2_sgd = scaler_sgd.transform(X2)
X3_sgd = scaler_sgd.transform(X3)
X_test_sgd = scaler_sgd.transform(X_test)

sgd_model = SGDRegressor(random_state=42)

# Iteration 1
sgd_model.partial_fit(X1_sgd, y1)
sgd_pred1 = sgd_model.predict(X1_sgd)
mse, rmse, mae, r2, acc = calculate_metrics(y1, sgd_pred1)
results.append(["Iteration 1", "Linear (SGD)", rmse])
all_iteration_metrics.append(["Iteration 1", "Linear (SGD)", mse, rmse, mae, r2, acc])

# Iteration 2
sgd_model.partial_fit(X2_sgd, y2)
sgd_pred2 = sgd_model.predict(X2_sgd)
mse, rmse, mae, r2, acc = calculate_metrics(y2, sgd_pred2)
results.append(["Iteration 2", "Linear (SGD)", rmse])
all_iteration_metrics.append(["Iteration 2", "Linear (SGD)", mse, rmse, mae, r2, acc])

# Iteration 3
sgd_model.partial_fit(X3_sgd, y3)
sgd_pred3 = sgd_model.predict(X3_sgd)
mse, rmse, mae, r2, acc = calculate_metrics(y3, sgd_pred3)
results.append(["Iteration 3", "Linear (SGD)", rmse])
all_iteration_metrics.append(["Iteration 3", "Linear (SGD)", mse, rmse, mae, r2, acc])

# Final test
sgd_test_pred = sgd_model.predict(X_test_sgd)
mse, rmse, mae, r2, acc = calculate_metrics(y_test, sgd_test_pred)
final_test_results.append(["Linear (SGD)", mse, rmse, mae, r2, acc])

In [26]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, Flatten, Dense, Input

scaler_cnn = StandardScaler()

X1_cnn = scaler_cnn.fit_transform(X1)
X2_cnn = scaler_cnn.transform(X2)
X3_cnn = scaler_cnn.transform(X3)
X_test_cnn = scaler_cnn.transform(X_test)

X1_cnn = X1_cnn.reshape((X1_cnn.shape[0], X1_cnn.shape[1], 1))
X2_cnn = X2_cnn.reshape((X2_cnn.shape[0], X2_cnn.shape[1], 1))
X3_cnn = X3_cnn.reshape((X3_cnn.shape[0], X3_cnn.shape[1], 1))
X_test_cnn = X_test_cnn.reshape((X_test_cnn.shape[0], X_test_cnn.shape[1], 1))

cnn_model = Sequential([
    Input(shape=(X1_cnn.shape[1], 1)),
    Conv1D(filters=32, kernel_size=2, activation='relu', padding='same'),
    Flatten(),
    Dense(64, activation='relu'),
    Dense(1)
])

cnn_model.compile(optimizer='adam', loss='mse')

# Iteration 1
cnn_model.fit(X1_cnn, y1, epochs=10, batch_size=32, verbose=0)
cnn_pred1 = cnn_model.predict(X1_cnn, verbose=0).flatten()
mse, rmse, mae, r2, acc = calculate_metrics(y1, cnn_pred1)
results.append(["Iteration 1", "CNN", rmse])
all_iteration_metrics.append(["Iteration 1", "CNN", mse, rmse, mae, r2, acc])

# Iteration 2
cnn_model.fit(X2_cnn, y2, epochs=10, batch_size=32, verbose=0)
cnn_pred2 = cnn_model.predict(X2_cnn, verbose=0).flatten()
mse, rmse, mae, r2, acc = calculate_metrics(y2, cnn_pred2)
results.append(["Iteration 2", "CNN", rmse])
all_iteration_metrics.append(["Iteration 2", "CNN", mse, rmse, mae, r2, acc])

# Iteration 3
cnn_model.fit(X3_cnn, y3, epochs=10, batch_size=32, verbose=0)
cnn_pred3 = cnn_model.predict(X3_cnn, verbose=0).flatten()
mse, rmse, mae, r2, acc = calculate_metrics(y3, cnn_pred3)
results.append(["Iteration 3", "CNN", rmse])
all_iteration_metrics.append(["Iteration 3", "CNN", mse, rmse, mae, r2, acc])

# Final test
cnn_test_pred = cnn_model.predict(X_test_cnn, verbose=0).flatten()
mse, rmse, mae, r2, acc = calculate_metrics(y_test, cnn_test_pred)
final_test_results.append(["CNN", mse, rmse, mae, r2, acc])

In [27]:
# !pip install xgboost

import xgboost as xgb

xgb_model = xgb.XGBRegressor(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='reg:squarederror',
    random_state=42
)

# Iteration 1
xgb_model.fit(X1, y1)
xgb_pred1 = xgb_model.predict(X1)
mse, rmse, mae, r2, acc = calculate_metrics(y1, xgb_pred1)
results.append(["Iteration 1", "XGBoost", rmse])
all_iteration_metrics.append(["Iteration 1", "XGBoost", mse, rmse, mae, r2, acc])

# Iteration 2
xgb_model.fit(X2, y2, xgb_model=xgb_model.get_booster())
xgb_pred2 = xgb_model.predict(X2)
mse, rmse, mae, r2, acc = calculate_metrics(y2, xgb_pred2)
results.append(["Iteration 2", "XGBoost", rmse])
all_iteration_metrics.append(["Iteration 2", "XGBoost", mse, rmse, mae, r2, acc])

# Iteration 3
xgb_model.fit(X3, y3, xgb_model=xgb_model.get_booster())
xgb_pred3 = xgb_model.predict(X3)
mse, rmse, mae, r2, acc = calculate_metrics(y3, xgb_pred3)
results.append(["Iteration 3", "XGBoost", rmse])
all_iteration_metrics.append(["Iteration 3", "XGBoost", mse, rmse, mae, r2, acc])

# Final test
xgb_test_pred = xgb_model.predict(X_test)
mse, rmse, mae, r2, acc = calculate_metrics(y_test, xgb_test_pred)
final_test_results.append(["XGBoost", mse, rmse, mae, r2, acc])

In [28]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

# Iteration 1
rf_model.fit(X1, y1)
rf_pred1 = rf_model.predict(X1)
mse, rmse, mae, r2, acc = calculate_metrics(y1, rf_pred1)
results.append(["Iteration 1", "Random Forest", rmse])
all_iteration_metrics.append(["Iteration 1", "Random Forest", mse, rmse, mae, r2, acc])

# Iteration 2
X12 = pd.concat([X1, X2])
y12 = pd.concat([y1, y2])
rf_model.fit(X12, y12)
rf_pred2 = rf_model.predict(X2)
mse, rmse, mae, r2, acc = calculate_metrics(y2, rf_pred2)
results.append(["Iteration 2", "Random Forest", rmse])
all_iteration_metrics.append(["Iteration 2", "Random Forest", mse, rmse, mae, r2, acc])

# Iteration 3
X123 = pd.concat([X1, X2, X3])
y123 = pd.concat([y1, y2, y3])
rf_model.fit(X123, y123)
rf_pred3 = rf_model.predict(X3)
mse, rmse, mae, r2, acc = calculate_metrics(y3, rf_pred3)
results.append(["Iteration 3", "Random Forest", rmse])
all_iteration_metrics.append(["Iteration 3", "Random Forest", mse, rmse, mae, r2, acc])

# Final test
rf_test_pred = rf_model.predict(X_test)
mse, rmse, mae, r2, acc = calculate_metrics(y_test, rf_test_pred)
final_test_results.append(["Random Forest", mse, rmse, mae, r2, acc])

In [29]:
# !pip install lightgbm

import lightgbm as lgb

lgb_train1 = lgb.Dataset(X1, label=y1)
lgb_train2 = lgb.Dataset(X2, label=y2)
lgb_train3 = lgb.Dataset(X3, label=y3)

params = {
    "objective": "regression",
    "metric": "rmse",
    "learning_rate": 0.05,
    "num_leaves": 31,
    "verbose": -1
}

# Iteration 1
lgb_model = lgb.train(params, lgb_train1, num_boost_round=100)
lgb_pred1 = lgb_model.predict(X1)
mse, rmse, mae, r2, acc = calculate_metrics(y1, lgb_pred1)
results.append(["Iteration 1", "LightGBM", rmse])
all_iteration_metrics.append(["Iteration 1", "LightGBM", mse, rmse, mae, r2, acc])

# Iteration 2
lgb_model = lgb.train(params, lgb_train2, num_boost_round=100, init_model=lgb_model)
lgb_pred2 = lgb_model.predict(X2)
mse, rmse, mae, r2, acc = calculate_metrics(y2, lgb_pred2)
results.append(["Iteration 2", "LightGBM", rmse])
all_iteration_metrics.append(["Iteration 2", "LightGBM", mse, rmse, mae, r2, acc])

# Iteration 3
lgb_model = lgb.train(params, lgb_train3, num_boost_round=100, init_model=lgb_model)
lgb_pred3 = lgb_model.predict(X3)
mse, rmse, mae, r2, acc = calculate_metrics(y3, lgb_pred3)
results.append(["Iteration 3", "LightGBM", rmse])
all_iteration_metrics.append(["Iteration 3", "LightGBM", mse, rmse, mae, r2, acc])

# Final test
lgb_test_pred = lgb_model.predict(X_test)
mse, rmse, mae, r2, acc = calculate_metrics(y_test, lgb_test_pred)
final_test_results.append(["LightGBM", mse, rmse, mae, r2, acc])

In [30]:
all_iteration_metrics_df = pd.DataFrame(
    all_iteration_metrics,
    columns=["Iteration", "Model", "MSE", "RMSE", "MAE", "R2", "Accuracy (%)"]
).round(3)

all_iteration_metrics_df

,Iteration,Model,MSE,RMSE,MAE,R2,Accuracy (%)
0,Iteration 1,Linear (SGD),578.058,24.043,12.017,0.067,6.741
1,Iteration 2,Linear (SGD),196.454,14.016,11.109,0.647,64.700
2,Iteration 3,Linear (SGD),171.445,13.094,10.346,0.694,69.428
3,Iteration 1,CNN,96.465,9.822,7.356,0.844,84.437
4,Iteration 2,CNN,113.121,10.636,8.011,0.797,79.674
5,Iteration 3,CNN,87.312,9.344,7.026,0.844,84.431
6,Iteration 1,XGBoost,66.330,8.144,6.060,0.893,89.299
7,Iteration 2,XGBoost,78.042,8.834,6.605,0.860,85.977
8,Iteration 3,XGBoost,57.790,7.602,5.681,0.897,89.695
9,Iteration 1,Random Forest,7.559,2.749,1.929,0.988,98.780


In [31]:
results_df = pd.DataFrame(results, columns=["Iteration", "Model", "RMSE"])

comparison_table = results_df.pivot(
    index="Iteration",
    columns="Model",
    values="RMSE"
)

comparison_table = comparison_table.reindex(["Iteration 1", "Iteration 2", "Iteration 3"]).round(3)

comparison_table

Model,CNN,LightGBM,Linear (SGD),Random Forest,XGBoost
Iteration,,,,,
Iteration 1,9.822,8.307,24.043,2.749,8.144
Iteration 2,10.636,8.928,14.016,3.216,8.834
Iteration 3,9.344,7.785,13.094,2.897,7.602


In [32]:
final_test_df = pd.DataFrame(
    final_test_results,
    columns=["Model", "MSE", "RMSE", "MAE", "R2", "Accuracy (%)"]
).round(3)

final_test_df

,Model,MSE,RMSE,MAE,R2,Accuracy (%)
0,Linear (SGD),259.004,16.094,12.030,0.460,45.972
1,CNN,213.449,14.610,10.551,0.555,55.474
2,XGBoost,168.184,12.969,9.016,0.649,64.917
3,Random Forest,174.049,13.193,9.039,0.637,63.693
4,LightGBM,169.728,13.028,9.036,0.646,64.595


In [33]:
from google.colab import files

all_iteration_metrics_df.to_csv("humidity_training_iteration_metrics_all_models.csv", index=False)
comparison_table.to_csv("humidity_iteration_comparison_table.csv")
final_test_df.to_csv("humidity_final_test_results.csv", index=False)

files.download("humidity_training_iteration_metrics_all_models.csv")
files.download("humidity_iteration_comparison_table.csv")
files.download("humidity_final_test_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [34]:
train_df.to_csv("humidity_training_dataset.csv", index=False)
test_df.to_csv("humidity_testing_dataset.csv", index=False)

from google.colab import files
files.download("humidity_training_dataset.csv")
files.download("humidity_testing_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>